# ⚔️ 熱血動漫 RAG 智慧問答系統（海賊王・火影忍者・咒術迴戰）

本系統為 **Retrieval-Augmented Generation（檢索增強生成）** 實踐應用：
- **向量檢索**：使用 Google `EmbeddingGemma-2` 進行語意向量編碼，並由 `FAISS` 進行精確相似度檢索。
- **大語言模型**：透過 `AIsuite` 串接 `Groq` 平台上的 `gpt-oss-120b`，嚴格依據檢索到的參考文本生成精準答案。
- **互動展示**：以 `Gradio` 打造現代化響應式介面，直觀展示生成回答、檢索相似度排行與官方引用來源。

## 1. 安裝必要套件
安裝語意向量模型（`sentence-transformers`）、向量索引（`faiss-cpu`）、模型呼叫庫（`aisuite`, `groq`）以及網頁介面套件（`gradio`）。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1" "aisuite[groq]==0.2.0" "groq==0.37.1" "gradio==6.30.0"

## 2. 匯入套件與全域參數設定
設定推論硬體（自動偵測 GPU/CPU 與最佳精度）、指定 LLM 模型與預設檢索篇數（Top-K）。已移除不必要的繪圖模組，保持環境精簡。

In [ ]:
import os
import io
import re
import json
import html
import hashlib
import tempfile
import zipfile
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import torch
import faiss
import numpy as np
import pandas as pd
import gradio as gr
import aisuite as ai
from sentence_transformers import SentenceTransformer

# 全域模型與推論設定
LLM_MODEL = "groq:openai/gpt-oss-120b"
DEFAULT_TOP_K = 3
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 4096

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

print(f"✅ 運算裝置：{DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print(f"✅ 浮點精度：{DTYPE}")
print(f"✅ 生成模型：{LLM_MODEL}")

## 3. 從 GitHub 自動下載向量知識庫 (`vector_embedding.zip`)
由 GitHub 自動取得建庫階段已封裝的向量知識庫檔案，自動解壓出核心的 4 項資產：
- `index.faiss`：已建立完成的高維度向量搜尋索引
- `documents.json`：各文本的正文內容與詮釋資料（Metadata）
- `index_doc_ids.json`：向量索引順序與文件編號的精確對映
- `config.json`：建庫時使用的前綴規範、維度與版本配置記錄

In [ ]:
GITHUB_KB_URLS = [
    "https://raw.githubusercontent.com/Xiangggg0920/Generative-AI/main/vector_embedding.zip",
    "https://raw.githubusercontent.com/Xiangggg0920/Generative-AI/master/vector_embedding.zip",
]

archive_bytes = None
last_error = None

for url in GITHUB_KB_URLS:
    try:
        print(f"🔄 正在從 GitHub 下載向量知識庫：{url} ...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            archive_bytes = resp.read()
            print(f"✨ 下載完成！檔案大小：{len(archive_bytes):,} bytes")
            break
    except Exception as e:
        last_error = e

if archive_bytes is None:
    raise RuntimeError(f"❌ 下載失敗，請檢查儲存庫與檔案路徑：{last_error}")

required_files = {"index.faiss", "documents.json", "index_doc_ids.json", "config.json"}
KB_DIR = Path(tempfile.mkdtemp(prefix="rag_kb_"))

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    names = set(archive.namelist())
    missing = required_files - names
    if missing:
        raise ValueError(f"❌ 知識庫壓縮檔格式不完整，缺少：{missing}")
    for name in required_files:
        (KB_DIR / name).write_bytes(archive.read(name))

print("✅ 知識庫檔案校驗無誤：", ", ".join(sorted(required_files)))

## 4. 驗證資料完整性並載入 FAISS 向量索引
比對檔案 SHA-256 雜湊值防止資料損毀，載入向量索引矩陣與 29 篇動漫知識庫文件。

In [ ]:
config = json.loads((KB_DIR / "config.json").read_text(encoding="utf-8"))

# 檢查檔案雜湊完整性
for name in ["index.faiss", "documents.json", "index_doc_ids.json"]:
    expected_hash = config.get("file_sha256", {}).get(name)
    actual_hash = hashlib.sha256((KB_DIR / name).read_bytes()).hexdigest()
    if expected_hash and actual_hash != expected_hash:
        raise ValueError(f"❌ 檔案完整性檢查失敗：{name}")

# 載入正文與對映編號
documents = json.loads((KB_DIR / "documents.json").read_text(encoding="utf-8"))
index_doc_ids = json.loads((KB_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
documents_by_id = {doc["doc_id"]: doc for doc in documents}

# 載入 FAISS 索引
index = faiss.read_index(str(KB_DIR / "index.faiss"))
DIMENSION = config["dimension"]
QUERY_TEMPLATE = config["query_template"]
TOKEN_LIMIT = config.get("max_seq_length", 2048)

print(f"✅ 知識庫文件數：{index.ntotal} 篇")
print(f"✅ 向量維度：{DIMENSION}")
print(f"✅ 基準 Embedding 模型版本：{config['model_id']} ({config['model_revision'][:10]}...)")

# 預覽載入的知識庫文件概況
df_docs = pd.DataFrame([
    {
        "文件編號": d["doc_id"],
        "標題": d["metadata"].get("title", "未命名"),
        "字數": len(d["text"]),
        "來源網址": d["metadata"].get("source_url", "未提供")
    }
    for d in documents
])
display(df_docs.head(8))

## 5. 載入 Embedding 模型並定義向量檢索函式
使用與建庫完全一致的 `EmbeddingGemma-2` 模型為使用者的輸入問題編碼，透過 FAISS 執行餘弦相似度（Cosine Similarity）精確搜尋，挑選出最相關的前 Top-K 篇文件。

In [ ]:
try:
    model = SentenceTransformer(
        config["model_id"],
        revision=config["model_revision"],
        token=False,
        device=DEVICE,
        config_kwargs=config.get("model_config_kwargs", {}),
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
    model.max_seq_length = TOKEN_LIMIT
except Exception as exc:
    raise RuntimeError(f"❌ Embedding 模型載入失敗：{exc}") from exc

def embed_question(question: str):
    """將問題文字格式化加上任務前綴並轉換為 L2 正則化向量"""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    vector = model.encode(
        [query_input],
        prompt=config.get("encode_prompt", ""),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    vector = np.ascontiguousarray(vector, dtype=np.float32)
    faiss.normalize_L2(vector)
    return vector

def search_documents(question: str, top_k: int = 3):
    """從 FAISS 索引搜尋相似度前 top_k 的參考文件"""
    query_vector = embed_question(question)
    k = min(max(1, int(top_k)), index.ntotal)
    scores, positions = index.search(query_vector, k)
    
    results = []
    for rank, (pos, score) in enumerate(zip(positions[0], scores[0]), start=1):
        if pos < 0 or pos >= len(index_doc_ids):
            continue
        doc_id = index_doc_ids[int(pos)]
        doc = documents_by_id[doc_id]
        results.append({
            "rank": rank,
            "score": float(score),
            "doc_id": doc_id,
            "text": doc["text"],
            "metadata": doc["metadata"],
        })
    return results

def retrieval_table(results):
    """將檢索結果格式化為供介面展示的 DataFrame"""
    return pd.DataFrame([
        {
            "排名": r["rank"],
            "文件編號": r["doc_id"],
            "標題": r["metadata"].get("title", "未標示"),
            "相似度 (Cosine)": f"{r['score']:.4f}",
            "來源連結": r["metadata"].get("source_url") or "無"
        }
        for r in results
    ], columns=["排名", "文件編號", "標題", "相似度 (Cosine)", "來源連結"])

print(f"✅ Embedding 模型載入完成！最大 Context 限制：{TOKEN_LIMIT} tokens")

## 6. 初始化 Groq LLM 客戶端
從 Colab Secrets（左側鑰匙圖示）中安全讀取 `Groq` 密鑰，並初始化 AIsuite Client，金鑰內容全程安全保護不外洩。

In [ ]:
from google.colab import userdata

client = None
os.environ.pop("GROQ_API_KEY", None)

try:
    groq_api_key = userdata.get("Groq")
    if not groq_api_key or not groq_api_key.strip():
        raise ValueError("密鑰為空")
    os.environ["GROQ_API_KEY"] = groq_api_key.strip()
    del groq_api_key
    client = ai.Client(provider_configs={"groq": {"timeout": 90.0, "max_retries": 1}})
    print("✅ Groq API 金鑰讀取成功，AIsuite 用戶端已初始化完成！")
except Exception:
    os.environ.pop("GROQ_API_KEY", None)
    raise RuntimeError(
        "❌ 無法讀取 Groq 金鑰！請至 Colab 左側『密鑰 (Secrets)』新增名稱為『Groq』的 API Key，並開啟本筆記本存取權限。"
    ) from None

## 7. 定義 RAG 核心生成邏輯與來源排版函式
設計防範幻覺的 System Prompt，要求 LLM 僅本於檢索文本回答並引用文件編號；同時定義卡片式 HTML 來源展示組件。已修正原本引發 `NameError` 的孤立除錯變數。

In [ ]:
SYSTEM_PROMPT = """你是熱門熱血動漫（海賊王、火影忍者、咒術迴戰）的專業 RAG 智慧知識助理。你必須嚴格遵守以下規則：
1. 【嚴格憑據原則】：你的回答必須完全基於提供的參考資料內容，不得擅自憑空想像、推測漫畫未公開劇情或摻雜其他作品設定。
2. 【標註來源引用】：回答中提及的重要設定或關鍵情節，請在句末附上對應的文件編號標籤，例如 [doc_007] 或 [doc_028]。
3. 【庫外誠實拒答】：若問題不在提供的參考文件範圍內或資料不足以解答，請明確說明「目前知識庫沒有足夠資訊可回答此問題」，並指出缺少哪些設定。
4. 【條理分明繁中】：使用通順流暢的繁體中文，善用列點排版，讓讀者一目了然。
"""

def safe_api_error(exc):
    err = str(exc).lower()
    if any(w in err for w in ["401", "unauthorized", "api_key"]):
        return "⚠️ Groq API 驗證失敗，請檢查 Colab Secrets 中的金鑰設定。"
    if any(w in err for w in ["429", "rate limit", "quota"]):
        return "⚠️ API 請求頻率過高或額度已達上限，請稍候重試。"
    return "⚠️ 語言模型生成失敗，請檢查網路連線或 API 狀態。"

def generate_answer(question: str, results: list):
    """將檢索文件組合成 Context 並呼叫 Groq 模型生成回答"""
    if not results:
        return "目前知識庫沒有找到與此問題相關的參考文件。"
    
    context_json = json.dumps([
        {"doc_id": r["doc_id"], "text": r["text"], "metadata": r["metadata"]}
        for r in results
    ], ensure_ascii=False, indent=2)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"【參考文件】：\n{context_json}\n\n【使用者問題】：\n{question}"}
    ]
    
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        return response.choices[0].message.content.strip()
    except Exception as exc:
        raise RuntimeError(safe_api_error(exc)) from exc

def citation_note(answer: str, results: list):
    allowed = {r["doc_id"] for r in results}
    cited = set(re.findall(r"\[(doc_[A-Za-z0-9_-]+)\]", answer))
    if not cited:
        return "💡 提示：模型未標註文件引用（若為拒答則屬正常現象）。"
    if cited - allowed:
        return "⚠️ 警告：回答中出現未在檢索結果內的文件編號，請核對依據。"
    return "✅ 引用校驗：所有標註之文件編號均來自本次檢索結果。"

def source_link(metadata):
    url = metadata.get("source_url")
    if not url or not isinstance(url, str):
        return "<span style='color:#9ca3af;'>未提供官方連結</span>"
    return f'<a href="{html.escape(url, quote=True)}" target="_blank" style="color:#3b82f6; text-decoration:none; font-weight:600;" rel="noopener noreferrer">🔗 官方百科出處</a>'

def render_sources_cards(results):
    """將檢索到的資料渲染為精美的現代化卡片 HTML"""
    if not results:
        return "<div style='color:#9ca3af; padding:10px;'>尚未進行檢索。</div>"
    
    cards = []
    for r in results:
        m = r["metadata"]
        title = html.escape(str(m.get("title") or "未標示"))
        doc_id = html.escape(r["doc_id"])
        score = r["score"]
        category = html.escape(str(m.get("category") or "動漫百科"))
        link = source_link(m)
        text_preview = html.escape(r["text"])
        
        cards.append(f"""
        <div style="background: rgba(255, 255, 255, 0.05); border: 1px solid rgba(255, 255, 255, 0.12); border-radius: 12px; padding: 14px; margin-bottom: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom: 8px;">
                <span style="background: linear-gradient(135deg, #6366f1, #8b5cf6); color:white; padding: 3px 8px; border-radius: 6px; font-size: 12px; font-weight:bold;">{doc_id}</span>
                <span style="font-size: 13px; color: #10b981; font-weight:bold;">相似度：{score:.4f}</span>
            </div>
            <div style="font-size: 15px; font-weight: bold; margin-bottom: 6px;">{title} <span style="font-size:12px; font-weight:normal; color:#9ca3af;">({category})</span></div>
            <div style="font-size: 13px; color: #d1d5db; line-height: 1.5; max-height: 120px; overflow-y: auto; background: rgba(0,0,0,0.2); padding: 8px; border-radius: 6px; margin-bottom: 8px; white-space: pre-wrap;">{text_preview}</div>
            <div style="font-size: 12px;">{link}</div>
        </div>
        """)
    return "".join(cards)

def gradio_answer(question: str, top_k: int):
    empty_df = retrieval_table([])
    if not question or not question.strip():
        return "請先輸入想要查詢的問題！", empty_df, "<div style='color:#9ca3af;'>等待輸入...</div>", "⚠️ 請輸入問題"
    
    try:
        retrieved = search_documents(question, top_k)
    except Exception as exc:
        return f"檢索發生錯誤：{exc}", empty_df, "", "❌ 檢索失敗"
    
    try:
        answer = generate_answer(question, retrieved)
        status = citation_note(answer, retrieved)
    except Exception as exc:
        answer = f"回答生成失敗：{exc}"
        status = "❌ 生成終止"
        
    return answer, retrieval_table(retrieved), render_sources_cards(retrieved), status

print("✅ RAG 核心生成與渲染函式就緒！")

## 8. 建立美化版 Gradio 動漫問答介面
採用雙欄（左右分欄）卡片式現代化佈局，搭配自訂深色高質感主題：
- **左側控制台**：多行提問框、Top-K 篇數滑桿、快捷動作按鈕與預設精選熱門範例
- **右側生成區**：Markdown 渲染回答卡片、來源驗證狀態徽章、檢索相似度表格與摺疊式來源全文卡片

In [ ]:
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass

# 自訂優雅現代感主題
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Noto Sans TC"), "sans-serif"]
)

custom_css = """
.gradio-container { max-width: 1200px !important; margin: 0 auto !important; }
.hero-banner { text-align: center; padding: 20px 10px; margin-bottom: 15px; border-radius: 14px; background: linear-gradient(135deg, rgba(99, 102, 241, 0.15), rgba(139, 92, 246, 0.15)); border: 1px solid rgba(99, 102, 241, 0.25); }
.hero-title { font-size: 28px; font-weight: 800; background: linear-gradient(90deg, #6366f1, #a855f7, #ec4899); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin-bottom: 8px; }
.hero-subtitle { font-size: 14px; color: #94a3b8; }
.card-panel { border-radius: 12px !important; }
"""

with gr.Blocks(title="⚔️ 熱血動漫 RAG 知識庫助理", theme=custom_theme, css=custom_css) as demo:
    with gr.Column(elem_classes=["hero-banner"]):
        gr.HTML("""
            <div class="hero-title">⚔️ 熱血動漫 RAG 智慧問答庫</div>
            <div class="hero-subtitle">全面索引《海賊王》・《火影忍者》・《咒術迴戰》29 篇世界觀與主要角色深度解析</div>
        """)
    
    with gr.Row():
        # 左側：問題輸入與控制
        with gr.Column(scale=5):
            question_box = gr.Textbox(
                label="💬 請輸入動漫問題",
                placeholder="例如：五條悟的無下限術式原理是什麼？ 或 索隆的三把佩刀是哪些？",
                lines=4,
                max_lines=6
            )
            top_k_slider = gr.Slider(
                minimum=1,
                maximum=min(5, index.ntotal),
                value=min(DEFAULT_TOP_K, index.ntotal),
                step=1,
                label="🎯 檢索篇數 (Top-K)",
                info="設定每次檢索回傳的最相關文件數量"
            )
            with gr.Row():
                ask_btn = gr.Button("🚀 搜尋並回答", variant="primary", scale=2)
                clear_btn = gr.Button("🔄 清除輸入", variant="secondary", scale=1)
            
            gr.Markdown("#### 💡 熱門推薦問題點擊試試：")
            gr.Examples(
                examples=[
                    ["五條悟的無下限術式原理是什麼？", 3],
                    ["草帽海賊團中負責解讀歷史本文與掌舵的成員分別是誰？", 3],
                    ["兩面宿儺的斬擊「解」與「捌」有何差異？", 3],
                    ["寫輪眼的完整進化路徑為何？萬花筒寫輪眼過度使用有何副作用？", 3],
                    ["漩渦鳴人將風屬性查克拉注入螺旋丸開發出的招式是什麼？", 3],
                    ["妖精的尾巴主角納茲的魔法是什麼？", 3]
                ],
                inputs=[question_box, top_k_slider],
                label="推薦題目（含庫外拒答測試）"
            )

        # 右側：生成回答與佐證資料
        with gr.Column(scale=7):
            status_badge = gr.Textbox(label="系統即時狀態與引用檢驗", value="🟢 系統就緒，請在左側輸入問題", interactive=False)
            answer_box = gr.Markdown(label="🤖 AI 生成精準回答", value="*等待提問中...*")
            
            with gr.Tabs():
                with gr.TabItem("📑 參考文本來源卡片"):
                    sources_cards = gr.HTML("<div style='color:#9ca3af;'>尚未發動檢索</div>")
                with gr.TabItem("📊 向量檢索排行列表"):
                    table_box = gr.Dataframe(
                        headers=["排名", "文件編號", "標題", "相似度 (Cosine)", "來源連結"],
                        interactive=False,
                        wrap=True
                    )
    
    # 綁定互動事件
    ask_btn.click(
        fn=gradio_answer,
        inputs=[question_box, top_k_slider],
        outputs=[answer_box, table_box, sources_cards, status_badge],
        concurrency_limit=1,
        show_progress="full",
    )
    
    clear_btn.click(
        fn=lambda: ("", "🟢 已清除輸入", "*等待提問中...*", retrieval_table([]), "<div style='color:#9ca3af;'>尚未發動檢索</div>"),
        inputs=[],
        outputs=[question_box, status_badge, answer_box, table_box, sources_cards],
    )

print("✅ 美化版 Gradio 介面配置完成！請執行下一格啟動伺服器。")

## 9. 啟動 Gradio 互動展示伺服器
執行下方儲存格將啟動本地伺服器，並自動產生 72 小時有效的公開共享連結（`https://xxxx.gradio.live`），點擊即可在全螢幕瀏覽器中操作體驗。

In [ ]:
demo.queue(max_size=10, default_concurrency_limit=1)
demo.launch(share=True, inline=True, debug=False, show_error=False)

## 10. 停止展示服務
展示結束或需要重新載入介面時執行，安全關閉 Gradio 伺服器並釋放 Port。

In [ ]:
# 展示結束後執行本格關閉伺服器
if "demo" in globals():
    demo.close()
    print("🛑 Gradio 展示服務已成功停止。")